# 🏗️ ArchAI — Blender GPU Renderer (Kaggle)

GPU-рендер Blender для AI Architect.

**Архитектура:**
```
Gateway (Render) → ngrok → Kaggle Notebook (T4/P100) → Blender → GLB/PNG
```

**Лимиты Kaggle Free:**
- T4 GPU: ~30 часов/неделю
- Сессия: до 9 часов
- Автоотключение: 60 мин бездействия

In [ ]:
#@title 📦 Установка Blender + зависимости { display-mode: "form" }
!apt-get update -qq && apt-get install -y -qq blender > /dev/null 2>&1
!pip install -q flask flask-cors pyngrok httpx

import subprocess
result = subprocess.run(['blender', '--version'], capture_output=True, text=True)
print(result.stdout.strip())
print('✅ Blender установлен')

In [ ]:
#@title 🔑 Настройка ngrok { display-mode: "form" }
import os

# Вставьте ваш ngrok authtoken (бесплатный: https://dashboard.ngrok.com)
NGROK_TOKEN = ""  # @param {type:"string"}

if NGROK_TOKEN:
    os.environ['NGROK_TOKEN'] = NGROK_TOKEN
    !ngrok config add-authtoken $NGROK_TOKEN
    print('✅ ngrok настроен')
else:
    print('⚠️ NGROK_TOKEN не задан — будет использоваться polling-режим')

In [ ]:
#@title 🏠 Генератор зданий (bpy-скрипты) { display-mode: "form" }

BUILD_SCRIPT = '''
import bpy
import bmesh
import json
import os
import sys
import math

def clear_scene():
    bpy.ops.object.select_all(action='SELECT')
    bpy.ops.object.delete(use_global=False)
    for c in bpy.data.collections:
        for o in c.objects:
            bpy.data.objects.remove(o, do_unlink=True)

def create_material(name, color, roughness=0.5, metallic=0.0):
    mat = bpy.data.materials.new(name)
    mat.use_nodes = True
    bsdf = mat.node_tree.nodes.get('Principled BSDF')
    if bsdf:
        bsdf.inputs['Base Color'].default_value = color
        bsdf.inputs['Roughness'].default_value = roughness
        bsdf.inputs['Metallic'].default_value = metallic
    return mat

def create_box(name, loc, size, mat):
    bpy.ops.mesh.primitive_cube_add(size=1, location=loc)
    obj = bpy.context.active_object
    obj.name = name
    obj.scale = (size[0]/2, size[1]/2, size[2]/2)
    obj.data.materials.append(mat)
    bpy.ops.object.transform_apply(scale=True)
    return obj

def create_building(params):
    clear_scene()
    
    W = params.get('width', 10)
    L = params.get('length', 12)
    floors = params.get('floors', 2)
    fH = params.get('floor_height', 2.8)
    H = floors * fH
    mat_name = params.get('material', 'plaster')
    roof_type = params.get('roof_type', 'gabled')
    style = params.get('style', '')
    
    # Colors
    colors = {
        'brick': (0.65, 0.25, 0.15, 1),
        'wood': (0.55, 0.35, 0.2, 1),
        'glass': (0.7, 0.85, 0.9, 1),
        'plaster': (0.9, 0.88, 0.85, 1),
        'stone': (0.5, 0.48, 0.45, 1),
    }
    color = colors.get(mat_name, colors['plaster'])
    wall_mat = create_material('Wall', color, roughness=0.8)
    glass_mat = create_material('Glass', (0.8, 0.9, 0.95, 0.3), roughness=0.1, metallic=0.1)
    roof_mat = create_material('Roof', (0.3, 0.15, 0.08, 1), roughness=0.9)
    floor_mat = create_material('Floor', (0.6, 0.6, 0.6, 1), roughness=0.6)
    
    # Main building
    create_box('Building', (0, 0, H/2), (W, L, H), wall_mat)
    
    # Floor plates
    for i in range(floors + 1):
        z = i * fH
        create_box(f'Floor_{i}', (0, 0, z), (W+0.1, L+0.1, 0.15), floor_mat)
    
    # Windows
    for floor in range(floors):
        z = (floor + 0.5) * fH
        for side, axis, sign in [('front', 1, L/2), ('back', 1, -L/2)]:
            for wx in range(-int(W/3), int(W/3)+1, 2):
                if style == 'hitech':
                    create_box(f'Window_{side}_{floor}_{wx}', (wx, sign*1.01, z), (1.5, 0.05, 1.8), glass_mat)
                else:
                    create_box(f'Window_{side}_{floor}_{wx}', (wx, sign*1.01, z), (1.2, 0.05, 1.4), glass_mat)
    
    # Roof
    if roof_type == 'gabled':
        bpy.ops.mesh.primitive_cube_add(size=1, location=(0, 0, H + 1.5))
        roof = bpy.context.active_object
        roof.name = 'Roof'
        roof.scale = (W/2 + 0.5, L/2 + 0.5, 1.5)
        bpy.ops.object.transform_apply(scale=True)
        roof.data.materials.append(roof_mat)
        # Modify to gabled shape
        bm = bmesh.new()
        bm.from_mesh(roof.data)
        for v in bm.verts:
            if v.co.z > 0:
                if abs(v.co.y) < 0.1:
                    v.co.z += 1.0
        bm.to_mesh(roof.data)
        bm.free()
    elif roof_type == 'flat':
        create_box('Roof', (0, 0, H + 0.15), (W+0.3, L+0.3, 0.3), roof_mat)
    
    # Ground plane
    bpy.ops.mesh.primitive_plane_add(size=50, location=(0, 0, -0.01))
    ground = bpy.context.active_object
    ground.name = 'Ground'
    ground_mat = create_material('Ground', (0.15, 0.18, 0.12, 1), roughness=1.0)
    ground.data.materials.append(ground_mat)
    
    return {'status': 'ok', 'objects': len(bpy.data.objects)}

def create_interior(params):
    clear_scene()
    
    W = params.get('width', 5)
    L = params.get('length', 6)
    H = params.get('height', 2.8)
    room_type = params.get('room_type', 'living')
    style = params.get('style', '')
    furniture = params.get('furniture', [])
    
    # Walls
    wall_color = (0.95, 0.93, 0.9, 1) if style != 'hitech' else (0.9, 0.9, 0.92, 1)
    wall_mat = create_material('Wall', wall_color, roughness=0.8)
    floor_mat = create_material('Floor', (0.6, 0.5, 0.35, 1), roughness=0.5)
    
    # Floor
    create_box('Floor', (0, 0, -0.05), (W, L, 0.1), floor_mat)
    
    # Walls
    create_box('Wall_N', (0, L/2, H/2), (W, 0.1, H), wall_mat)
    create_box('Wall_S', (0, -L/2, H/2), (W, 0.1, H), wall_mat)
    create_box('Wall_E', (W/2, 0, H/2), (0.1, L, H), wall_mat)
    create_box('Wall_W', (-W/2, 0, H/2), (0.1, L, H), wall_mat)
    
    # Ceiling
    ceiling_mat = create_material('Ceiling', (1, 1, 1, 1), roughness=0.9)
    create_box('Ceiling', (0, 0, H), (W, L, 0.05), ceiling_mat)
    
    # Furniture
    wood_mat = create_material('Wood', (0.55, 0.35, 0.2, 1), roughness=0.6)
    fabric_mat = create_material('Fabric', (0.4, 0.4, 0.45, 1), roughness=0.9)
    metal_mat = create_material('Metal', (0.7, 0.7, 0.7, 1), roughness=0.3, metallic=0.8)
    
    furniture_defs = {
        'bed': lambda: create_box('Bed', (0, -L/4, 0.3), (2, 1.8, 0.6), fabric_mat),
        'sofa': lambda: create_box('Sofa', (0, -L/4, 0.3), (2.5, 1, 0.7), fabric_mat),
        'desk': lambda: create_box('Desk', (-W/3, L/3, 0.4), (1.2, 0.6, 0.8), wood_mat),
        'wardrobe': lambda: create_box('Wardrobe', (W/3, L/3, 0.9), (1.5, 0.6, 1.8), wood_mat),
        'table': lambda: create_box('Table', (0, 0, 0.4), (1.4, 0.8, 0.8), wood_mat),
        'chairs': lambda: create_box('Chair', (0.8, 0, 0.25), (0.4, 0.4, 0.5), wood_mat),
        'bookshelf': lambda: create_box('Bookshelf', (W/3, 0, 0.9), (1.2, 0.35, 1.8), wood_mat),
        'tv': lambda: create_box('TV', (0, L/2-0.1, 1.2), (1.2, 0.05, 0.7), metal_mat),
        'bathtub': lambda: create_box('Bathtub', (-W/4, 0, 0.3), (1.8, 0.8, 0.6), metal_mat),
        'sink': lambda: create_box('Sink', (W/4, L/3, 0.5), (0.6, 0.45, 0.3), metal_mat),
        'toilet': lambda: create_box('Toilet', (W/3, -L/3, 0.25), (0.4, 0.6, 0.5), (1,1,1,1)),
        'fridge': lambda: create_box('Fridge', (W/3, -L/3, 0.9), (0.7, 0.7, 1.8), metal_mat),
        'stove': lambda: create_box('Stove', (-W/3, -L/3, 0.45), (0.6, 0.6, 0.9), metal_mat),
    }
    
    placed = []
    for item in furniture:
        if item in furniture_defs:
            furniture_defs[item]()
            placed.append(item)
    
    return {'status': 'ok', 'objects': len(bpy.data.objects), 'furniture': placed}

if __name__ == '__main__' and len(sys.argv) > 2:
    params = json.loads(sys.argv[2])
    obj_type = params.get('object_type', 'building')
    if obj_type == 'interior' or obj_type == 'room':
        result = create_interior(params)
    else:
        result = create_building(params)
    
    output_path = sys.argv[3] if len(sys.argv) > 3 else '/tmp/output.glb'
    bpy.ops.export_scene.gltf(filepath=output_path, export_format='GLB')
    print(json.dumps(result))
'''

with open('/tmp/archai_render.py', 'w') as f:
    f.write(BUILD_SCRIPT)
print('✅ Генератор зданий создан')

In [ ]:
#@title 🖥️ Flask рендер-сервер { display-mode: "form" }

import subprocess
import threading
import time
import json
import os
import tempfile
import base64
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

RENDER_SCRIPT = '/tmp/archai_render.py'
RENDER_COUNT = 0
START_TIME = time.time()

@app.route('/health', methods=['GET'])
def health():
    gpu_info = ''
    try:
        gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader'], 
                                capture_output=True, text=True).stdout.strip()
    except: pass
    return jsonify({
        'status': 'ok',
        'service': 'kaggle-blender',
        'version': '1.0.0',
        'gpu': gpu_info,
        'renders': RENDER_COUNT,
        'uptime_seconds': int(time.time() - START_TIME)
    })

@app.route('/api/v1/generate', methods=['POST'])
def generate():
    global RENDER_COUNT
    try:
        data = request.json or {}
        prompt = data.get('prompt', '')
        params = data.get('params', data)
        
        # Merge prompt-derived params
        if prompt and not params.get('width'):
            params['prompt'] = prompt
        
        output_path = tempfile.mktemp(suffix='.glb')
        params_json = json.dumps(params, ensure_ascii=False)
        
        # Run Blender
        cmd = ['blender', '--background', '--python', RENDER_SCRIPT, '--', params_json, output_path]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        
        if result.returncode != 0:
            return jsonify({'error': 'blender_failed', 'stderr': result.stderr[-500:]}), 500
        
        # Parse Blender output
        render_result = {}
        for line in result.stdout.split('\n'):
            line = line.strip()
            if line.startswith('{'):
                try: render_result = json.loads(line)
                except: pass
        
        if not os.path.exists(output_path):
            return jsonify({'error': 'no_output', 'stdout': result.stdout[-500:]}), 500
        
        RENDER_COUNT += 1
        
        # Return GLB file
        return send_file(output_path, mimetype='model/gltf-binary', as_attachment=True,
                        download_name='model.glb')
    except subprocess.TimeoutExpired:
        return jsonify({'error': 'timeout'}), 504
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/v1/preview', methods=['POST'])
def preview():
    """Рендер превью-картинки (PNG)"""
    try:
        data = request.json or {}
        params = data.get('params', data)
        width = data.get('width', 1920)
        height = data.get('height', 1080)
        
        output_glb = tempfile.mktemp(suffix='.glb')
        output_png = tempfile.mktemp(suffix='.png')
        params_json = json.dumps(params, ensure_ascii=False)
        
        # Generate GLB first
        cmd = ['blender', '--background', '--python', RENDER_SCRIPT, '--', params_json, output_glb]
        subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        
        if not os.path.exists(output_glb):
            return jsonify({'error': 'no_model'}), 500
        
        # Render PNG with Blender
        render_script = f'''
import bpy
bpy.ops.import_scene.gltf(filepath="{output_glb}")
bpy.context.scene.render.resolution_x = {width}
bpy.context.scene.render.resolution_y = {height}
bpy.context.scene.render.resolution_percentage = 100
bpy.context.scene.render.film_transparent = False
# Camera
bpy.ops.object.camera_add(location=(15, -15, 12))
cam = bpy.context.active_object
cam.rotation_euler = (1.1, 0, 0.785)
bpy.context.scene.camera = cam
# Lighting
bpy.ops.object.light_add(type='SUN', location=(10, -10, 20))
light = bpy.context.active_object
light.data.energy = 3.0
# Render
bpy.context.scene.render.filepath = "{output_png}"
bpy.ops.render.render(write_still=True)
'''
        render_path = tempfile.mktemp(suffix='.py')
        with open(render_path, 'w') as f:
            f.write(render_script)
        
        subprocess.run(['blender', '--background', '--python', render_path], 
                      capture_output=True, text=True, timeout=180)
        
        if os.path.exists(output_png):
            return send_file(output_png, mimetype='image/png')
        return jsonify({'error': 'render_failed'}), 500
    except Exception as e:
        return jsonify({'error': str(e)}), 500

def run_server():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print('✅ Flask сервер запущен на порту 5000')

In [ ]:
#@title 🌐 Запуск ngrok-туннеля { display-mode: "form" }

from pyngrok import ngrok

NGROK_TOKEN = ""  # @param {type:"string"}
PUBLIC_URL = None

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(5000, bind_tls=True)
    PUBLIC_URL = tunnel.public_url
    print(f'✅ Публичный URL: {PUBLIC_URL}')
    print(f'\nДобавьте в Render env:')
    print(f'KAGGLE_RENDERER_URL={PUBLIC_URL}')
else:
    print('⚠️ NGROK_TOKEN не задан')
    print('Получите бесплатный токен: https://dashboard.ngrok.com/get-started/your-authtoken')
    print('Или используйте polling-режим (см. следующую ячейку)')

In [ ]:
#@title 🔄 Polling-режим (без ngrok) { display-mode: "form" }

# Если ngrok не настроен, ноутбук будет опрашивать Gateway на наличие задач

import httpx
import time

GATEWAY_URL = "https://architect-gateway.onrender.com"  # @param {type:"string"}
POLL_INTERVAL = 10  # секунд

def poll_for_tasks():
    print(f'🔄 Polling {GATEWAY_URL} every {POLL_INTERVAL}s...')
    print('Остановите ячейку для завершения')
    
    while True:
        try:
            # Check if there's a pending render task
            r = httpx.get(f'{GATEWAY_URL}/api/v1/orchestrator/pending-kaggle', timeout=15)
            if r.status_code == 200:
                task = r.json()
                if task and task.get('prompt'):
                    print(f'📥 Получена задача: {task["prompt"][:60]}...')
                    # Render
                    result = render_task(task)
                    # Upload result back
                    if result:
                        httpx.post(f'{GATEWAY_URL}/api/v1/orchestrator/kaggle-result', 
                                  json={'task_id': task.get('id'), 'result': result}, timeout=30)
                        print(f'✅ Результат отправлен')
        except httpx.ConnectError:
            pass  # Gateway sleeping
        except Exception as e:
            print(f'⚠️ Poll error: {e}')
        
        time.sleep(POLL_INTERVAL)

def render_task(task):
    """Рендер задачи и возврат base64 GLB"""
    import subprocess, tempfile, json, base64, os
    
    output = tempfile.mktemp(suffix='.glb')
    params = task.get('params', {})
    params['prompt'] = task.get('prompt', '')
    
    cmd = ['blender', '--background', '--python', '/tmp/archai_render.py', '--', 
           json.dumps(params, ensure_ascii=False), output]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    
    if os.path.exists(output):
        with open(output, 'rb') as f:
            return base64.b64encode(f.read()).decode()
    return None

if not PUBLIC_URL:
    poll_for_tasks()

In [ ]:
#@title 🧪 Тест рендера { display-mode: "form" }

import requests
import json

test_url = PUBLIC_URL or 'http://localhost:5000'

# Тест 1: Health
try:
    r = requests.get(f'{test_url}/health', timeout=10)
    print(f'Health: {r.json()}')
except Exception as e:
    print(f'Health error: {e}')

# Тест 2: Генерация здания
try:
    r = requests.post(f'{test_url}/api/v1/generate', json={
        'prompt': 'двухэтажный кирпичный дом 10x12',
        'object_type': 'building',
        'floors': 2, 'width': 10, 'length': 12,
        'material': 'brick', 'roof_type': 'gabled'
    }, timeout=120)
    if r.status_code == 200:
        with open('/tmp/test_building.glb', 'wb') as f:
            f.write(r.content)
        print(f'✅ Здание: {len(r.content)} bytes')
    else:
        print(f'❌ Здание: {r.status_code} {r.text[:200]}')
except Exception as e:
    print(f'❌ Здание: {e}')

# Тест 3: Генерация интерьера
try:
    r = requests.post(f'{test_url}/api/v1/generate', json={
        'prompt': 'ванная в стиле хайтек с джакузи',
        'object_type': 'interior',
        'room_type': 'bathroom', 'width': 4, 'length': 5, 'height': 2.8,
        'style': 'hitech', 'furniture': ['bathtub', 'sink', 'toilet']
    }, timeout=120)
    if r.status_code == 200:
        with open('/tmp/test_interior.glb', 'wb') as f:
            f.write(r.content)
        print(f'✅ Интерьер: {len(r.content)} bytes')
    else:
        print(f'❌ Интерьер: {r.status_code} {r.text[:200]}')
except Exception as e:
    print(f'❌ Интерьер: {e}')

In [ ]:
#@title ♾️ Keep-alive (предотвращение автоотключения) { display-mode: "form" }

KEEP_ALIVE = True  # @param {type:"boolean"}

if KEEP_ALIVE and PUBLIC_URL:
    print(f'🔄 Keep-alive: pinging {PUBLIC_URL}/health every 5 min')
    while True:
        try:
            requests.get(f'{PUBLIC_URL}/health', timeout=5)
        except: pass
        time.sleep(300)